# 量化交易机器学习策略 — 完整实战

**目标**: 基于存储的模型样本数据，构建多模型交易策略，完成回测与模型对比

**流程**:
1. 加载存储的模型样本
2. 衍生自变量因子 + 设计因变量
3. 划分训练集/测试集，训练模型
4. 建立交易策略，计算季度收益率
5. 回测策略，计算核心指标，绘制图形
6. 对比决策树、随机森林等模型效果

## 1. 加载存储的模型样本

In [ ]:
print("1. 加载存储的模型样本")
model_df = pd.read_csv(os.path.join(BASE_DIR, 'model_data.csv'))
print(f'model_data.csv: {model_df.shape[0]} rows, {model_df.shape[1]} cols')
print(f'Columns: {list(model_df.columns)[:5]}...')
print()
daily_600750 = pd.read_csv(os.path.join(BASE_DIR, '600750_daily_fq.csv'))
daily_000538 = pd.read_csv(os.path.join(BASE_DIR, '000538_daily_fq.csv'))
print(f'600750 daily: {daily_600750.shape[0]} rows')
print(f'000538 daily: {daily_000538.shape[0]} rows')
# 基本探索
model_df['Date'] = pd.to_datetime(model_df['Date'])
next_ret = model_df['Next_Ret'].dropna()
print(f'\nNext_Ret stats: mean={next_ret.mean():.4f}, std={next_ret.std():.4f}')
print(f'Positive ratio: {(next_ret > 0).mean():.2%}')
print(f'Time range: {model_df["Date"].min()} ~ {model_df["Date"].max()}')
print(f'Stocks: {model_df["Code"].nunique()}, Quarters: {model_df["Date"].nunique()}')

[Section 1 executed - see HTML dashboard for full results]


## 2. 衍生自变量因子 & 设计因变量

In [ ]:
print("2. 衍生自变量因子 & 设计因变量")
fundamental_cols = list(model_df.columns[2:-1])  # Exclude Date, Code, Next_Ret
print(f'Original fundamental factors: {len(fundamental_cols)}')
# 衍生因子: 横截面排名百分位
derived_df = model_df.copy()
for col in fundamental_cols:
    rank_col = f'{col}_rank'
    derived_df[rank_col] = derived_df.groupby('Date')[col].rank(pct=True)
# 复合因子
pb_rank_col = f'市净率PB(MRQ)_rank'
pe_rank_col = f'市盈率PE(TTM)_rank'
rev_rank_col = f'营业收入(同比增长率)_rank'
profit_rank_col = f'净利润同比增长率_rank'
mv_rank_col = f'MV_rank'
derived_df['value_score'] = derived_df[pb_rank_col] * 0.5 + derived_df[pe_rank_col] * 0.5
derived_df['growth_score'] = derived_df[rev_rank_col] * 0.5 + derived_df[profit_rank_col] * 0.5
derived_df['quality_score'] = derived_df[profit_rank_col]
derived_df['size_score'] = derived_df[mv_rank_col]
# 因变量定义
derived_df['label_binary'] = (derived_df['Next_Ret'] > 0).astype(int)
conditions = [
    derived_df['Next_Ret'] > 0.03,
    derived_df['Next_Ret'] < -0.03,
]
choices = [2, 0]
derived_df['label_multi'] = np.select(conditions, choices, default=1)
derived_df['label_rank'] = derived_df.groupby('Date')['Next_Ret'].transform(
    lambda x: pd.cut(x.rank(pct=True), bins=[0, 0.3, 0.7, 1.0], labels=[0, 1, 2])
).astype(int)
print(f'\nBinary label distribution:')
print(derived_df['label_binary'].value_counts())
print(f'\nMulti label distribution:')
print(derived_df['label_multi'].value_counts())
# 技术指标衍生 (以600750为例)
def compute_technical_factors(df):
    df = df.copy()
    df['trade_date'] = pd.to_datetime(df['trade_date'], format='%Y%m%d')
    df = df.sort_values('trade_date').reset_index(drop=True)
    close = df['close']
    for n in [1, 5, 10, 20]:
        df[f'ret_{n}d'] = close / close.shift(n) - 1
    for n in [5, 10, 20]:
        ma = close.rolling(n).mean()
        df[f'ma_{n}_bias'] = (close - ma) / ma
    df['vol_5d'] = df['pct_chg'].rolling(5).std()
    df['vol_20d'] = df['pct_chg'].rolling(20).std()
    df['vol_change_5d'] = df['vol'] / df['vol'].rolling(5).mean() - 1
    # RSI(14)
    delta = close.diff()
    gain = delta.where(delta > 0, 0).rolling(14).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(14).mean()
    rs = gain / loss
    df['RSI_14'] = 100 - (100 / (1 + rs))
    # MACD
    ema12 = close.ewm(span=12).mean()
    ema26 = close.ewm(span=26).mean()
    df['MACD'] = ema12 - ema26
    df['MACD_signal'] = df['MACD'].ewm(span=9).mean()
    df['MACD_hist'] = df['MACD'] - df['MACD_signal']
    # KDJ
    low_min = df['low'].rolling(9).min()
    high_max = df['high'].rolling(9).max()
    rsv = (close - low_min) / (high_max - low_min) * 100
    df['K'] = rsv.ewm(com=2).mean()
    df['D'] = df['K'].ewm(com=2).mean()
    df['J'] = 3 * df['K'] - 2 * df['D']
    # ATR
    tr = pd.concat([
        df['high'] - df['low'],
        (df['high'] - close.shift(1)).abs(),
        (df['low'] - close.shift(1)).abs()
    ], axis=1).max(axis=1)
    df['ATR_14'] = tr.rolling(14).mean()
    df['ATR_ratio'] = df['ATR_14'] / close
    # Bollinger Bands
    ma20 = close.rolling(20).mean()
    std20 = close.rolling(20).std()
    df['BB_width'] = 4 * std20 / ma20
    df['BB_pos'] = (close - (ma20 - 2*std20)) / (4 * std20)
    return df
tech_600750 = compute_technical_factors(daily_600750)
tech_cols = [c for c in tech_600750.columns if c not in daily_600750.columns]
print(f'\n600750 derived technical indicators: {len(tech_cols)}')
print(f'Tech factors: {tech_cols[:10]}...')
# ========== 3. 划分训练集/测试集，构建并训练模型 ==========
print("3. 划分训练集/测试集 & 训练模型")
rank_cols = [f'{col}_rank' for col in fundamental_cols]
composite_cols = ['value_score', 'growth_score', 'quality_score', 'size_score']
feature_cols = rank_cols + composite_cols
clean_df = derived_df.dropna(subset=feature_cols + ['label_binary', 'Next_Ret']).copy()
print(f'Cleaned data: {len(clean_df)} rows')
print(f'Features: {len(feature_cols)}')
# 时间序列划分
dates_sorted = sorted(clean_df['Date'].unique())
n_dates = len(dates_sorted)
split_idx = int(n_dates * 0.7)
train_dates = dates_sorted[:split_idx]
test_dates = dates_sorted[split_idx:]
train_mask = clean_df['Date'].isin(train_dates)
test_mask = clean_df['Date'].isin(test_dates)
X_train = clean_df.loc[train_mask, feature_cols]
X_test = clean_df.loc[test_mask, feature_cols]
y_train_binary = clean_df.loc[train_mask, 'label_binary']
y_test_binary = clean_df.loc[test_mask, 'label_binary']
y_test_ret = clean_df.loc[test_mask, 'Next_Ret']
print(f'Train: {X_train.shape[0]} rows, {train_dates[0]} ~ {train_dates[-1]}')
print(f'Test: {X_test.shape[0]} rows, {test_dates[0]} ~ {test_dates[-1]}')
print(f'Train positive ratio: {y_train_binary.mean():.2%}')
print(f'Test positive ratio: {y_test_binary.mean():.2%}')
# 标准化
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
print('Feature scaling done')
# 构建并训练多个模型
models = {
    '决策树': DecisionTreeClassifier(max_depth=5, random_state=42),
    '随机森林': RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42),
    '逻辑回归': LogisticRegression(C=1.0, max_iter=1000, random_state=42),
    'GBDT': GradientBoostingClassifier(n_estimators=100, max_depth=4, random_state=42),
}
if HAS_XGB:
    models['XGBoost'] = xgb.XGBClassifier(
        n_estimators=100, max_depth=4, random_state=42,
        eval_metric='logloss', verbosity=0
    )
results = {}
for name, model in models.items():
    print(f'\n--- Training {name} ---')
    model.fit(X_train_scaled, y_train_binary)
    y_pred_prob = model.predict_proba(X_test_scaled)[:, 1]
    y_pred = model.predict(X_test_scaled)
    acc = accuracy_score(y_test_binary, y_pred)
    try:
        auc = roc_auc_score(y_test_binary, y_pred_prob)
    except:
        auc = 0.0
    print(f'  Accuracy: {acc:.4f}')
    print(f'  AUC: {auc:.4f}')
    results[name] = {
        'model': model,
        'y_pred_prob': y_pred_prob,
        'y_pred': y_pred,
        'accuracy': acc,
        'auc': auc,
    }
print('\nAll models trained!')
# ========== 4. 基于模型建立交易策略，计算季度收益率 ==========
print("4. 交易策略 & 季度收益率")
TOP_N = 20
test_df = clean_df.loc[test_mask].copy()
strategy_returns = {}
for model_name, res in results.items():
    test_df[f'prob_{model_name}'] = res['y_pred_prob']
    quarterly_returns = {}
    for date in test_dates:
        quarter_data = test_df[test_df['Date'] == date]
        if len(quarter_data) < TOP_N:
            continue
        top_stocks = quarter_data.nlargest(TOP_N, f'prob_{model_name}')
        avg_return = top_stocks['Next_Ret'].mean()
        quarterly_returns[date] = avg_return
    strategy_returns[model_name] = quarterly_returns
# 基准
benchmark_returns = {}
for date in test_dates:
    quarter_data = test_df[test_df['Date'] == date]
    if len(quarter_data) > 0:
        benchmark_returns[date] = quarter_data['Next_Ret'].mean()
strategy_returns['基准(全市场等权)'] = benchmark_returns
print('=== Quarterly Returns ===')
for model_name, qret in strategy_returns.items():
    print(f'\n{model_name}:')
    for date, ret in sorted(qret.items()):
        print(f'  {pd.Timestamp(date).strftime("%Y-Q%m")}: {ret:.4f} ({ret*100:.2f}%)')
    avg = np.mean(list(qret.values()))
    print(f'  Average: {avg:.4f} ({avg*100:.2f}%)')
# ========== 5. 回测策略，计算核心指标，绘制图形 ==========
print("5. 回测 & 核心指标")
def calc_backtest_metrics(returns_dict, name='Strategy'):
    returns = np.array(list(returns_dict.values()))
    dates = sorted(returns_dict.keys())
    nav = np.cumprod(1 + returns)
    total_return = nav[-1] - 1
    n_quarters = len(returns)
    annual_return = (1 + total_return) ** (4 / n_quarters) - 1 if n_quarters > 0 else 0
    annual_vol = np.std(returns) * np.sqrt(4)
    rf_per_quarter = 0.02 / 4
    sharpe = (np.mean(returns) - rf_per_quarter) / (np.std(returns) + 1e-10) * np.sqrt(4)
    peak = np.maximum.accumulate(nav)
    drawdown = (nav - peak) / peak
    max_drawdown = drawdown.min()
    calmar = annual_return / (-max_drawdown + 1e-10) if max_drawdown < 0 else float('inf')
    win_rate = (returns > 0).mean()
    avg_win = np.mean(returns[returns > 0]) if (returns > 0).any() else 0
    avg_loss = np.mean(returns[returns < 0]) if (returns < 0).any() else 0
    profit_loss_ratio = abs(avg_win / (avg_loss + 1e-10))
    return {
        'name': name,
        'total_return': total_return,
        'annual_return': annual_return,
        'annual_vol': annual_vol,
        'sharpe': sharpe,
        'max_drawdown': max_drawdown,
        'calmar': calmar,
        'win_rate': win_rate,
        'profit_loss_ratio': profit_loss_ratio,
        'avg_return_per_quarter': np.mean(returns),
        'nav': nav,
        'returns': returns,
        'dates': dates,
        'drawdown': drawdown,
    }
metrics = {}
for model_name, qret in strategy_returns.items():
    metrics[model_name] = calc_backtest_metrics(qret, model_name)
# 指标汇总
print('\n=== Backtest Metrics Summary ===')
summary_rows = []
for name, m in metrics.items():
    summary_rows.append({
        '模型': name,
        '累计收益': f'{m["total_return"]:.2%}',
        '年化收益': f'{m["annual_return"]:.2%}',
        '年化波动': f'{m["annual_vol"]:.2%}',
        '夏普比率': f'{m["sharpe"]:.2f}',
        '最大回撤': f'{m["max_drawdown"]:.2%}',
        'Calmar': f'{m["calmar"]:.2f}',
        '胜率': f'{m["win_rate"]:.2%}',
        '盈亏比': f'{m["profit_loss_ratio"]:.2f}',
    })
summary_df = pd.DataFrame(summary_rows)
print(summary_df.to_string(index=False))
# ========== 绘制图形 ==========
color_map = {
    '决策树': '#E24B4A', '随机森林': '#1D9E75', '逻辑回归': '#378ADD',
    'GBDT': '#BA7517', 'XGBoost': '#534AB7', '基准(全市场等权)': '#888780'
}
fig, axes = plt.subplots(3, 2, figsize=(14, 16))
# 累计净值
ax = axes[0, 0]
for name, m in metrics.items():
    c = color_map.get(name, '#888')
    dates_fmt = [pd.Timestamp(d).strftime('%Y-Q%m') for d in m['dates']]
    ax.plot(dates_fmt, m['nav'], label=name, color=c, linewidth=2)
ax.set_title('累计净值曲线对比', fontsize=14)
ax.set_xlabel('季度')
ax.set_ylabel('净值')
ax.legend(loc='upper left')
ax.grid(alpha=0.3)
ax.tick_params(axis='x', rotation=45)
# 季度收益率柱状图
ax = axes[0, 1]
model_names_plot = [n for n in metrics.keys() if n != '基准(全市场等权)']
test_dates_fmt = sorted(test_dates)
width = 0.15
for i, name in enumerate(model_names_plot):
    qret = strategy_returns[name]
    returns_vals = [qret.get(d, 0) * 100 for d in test_dates_fmt]
    dates_labels = [pd.Timestamp(d).strftime('%Y-Q%m') for d in test_dates_fmt]
    offset = (i - len(model_names_plot)/2) * width
    ax.bar([j + offset for j in range(len(returns_vals))], returns_vals,
           width=width, label=name, color=color_map.get(name, '#888'), alpha=0.8)
ax.set_title('季度收益率对比 (%)', fontsize=14)
ax.set_xlabel('季度')
ax.set_ylabel('收益率 (%)')
ax.legend()
ax.grid(alpha=0.3)
# 回撤曲线
ax = axes[1, 0]
for name, m in metrics.items():
    c = color_map.get(name, '#888')
    dates_fmt = [pd.Timestamp(d).strftime('%Y-Q%m') for d in m['dates']]
    ax.plot(dates_fmt, m['drawdown'] * 100, label=name, color=c, linewidth=1.5)
ax.set_title('回撤曲线对比 (%)', fontsize=14)
ax.set_xlabel('季度')
ax.set_ylabel('回撤 (%)')
ax.legend(loc='lower left')
ax.grid(alpha=0.3)
ax.tick_params(axis='x', rotation=45)
# AUC对比
ax = axes[1, 1]
model_eval_names = list(results.keys())
auc_vals = [results[n]['auc'] for n in model_eval_names]
acc_vals = [results[n]['accuracy'] for n in model_eval_names]
x_pos = range(len(model_eval_names))
bars1 = ax.bar(x_pos, auc_vals, width=0.35, label='AUC', color='#378ADD', alpha=0.8)
bars2 = ax.bar([p + 0.35 for p in x_pos], acc_vals, width=0.35, label='Accuracy', color='#1D9E75', alpha=0.8)
ax.set_xticks([p + 0.175 for p in x_pos])
ax.set_xticklabels(model_eval_names)
ax.set_title('模型分类性能对比', fontsize=14)
ax.set_ylabel('Score')
ax.legend()
ax.grid(alpha=0.3)
for bar in bars1 + bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{bar.get_height():.3f}', ha='center', fontsize=9)
# 夏普 vs 回撤
ax = axes[2, 0]
for name, m in metrics.items():
    c = color_map.get(name, '#888')
    ax.scatter(m['sharpe'], -m['max_drawdown']*100, s=120, color=c, label=name, zorder=5)
    ax.annotate(name, (m['sharpe'], -m['max_drawdown']*100), fontsize=9,
                textcoords='offset points', xytext=(8, 5))
ax.set_title('夏普比率 vs 最大回撤', fontsize=14)
ax.set_xlabel('夏普比率')
ax.set_ylabel('最大回撤 (%)')
ax.grid(alpha=0.3)
# 多指标对比
ax = axes[2, 1]
metric_keys = ['annual_return', 'sharpe', 'win_rate', 'profit_loss_ratio']
metric_labels = ['年化收益', '夏普比率', '胜率', '盈亏比']
for name, m in metrics.items():
    vals = [m[k] if k != 'win_rate' else m[k]*100 for k in metric_keys]
    ax.plot(metric_labels, vals, marker='o', label=name, linewidth=2)
ax.set_title('多指标对比', fontsize=14)
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(BASE_DIR, 'ml_strategy_backtest.png'), dpi=150, bbox_inches='tight')
print('Backtest chart saved to ml_strategy_backtest.png')
# 特征重要性图
fig2, axes2 = plt.subplots(2, 2, figsize=(14, 12))
tree_models = ['决策树', '随机森林', 'GBDT']
if HAS_XGB:
    tree_models.append('XGBoost')
for idx, name in enumerate(tree_models[:4]):
    if idx >= len(axes2.flat) or name not in results:
        continue
    ax = axes2.flat[idx]
    model = results[name]['model']
    if not hasattr(model, 'feature_importances_'):
        continue
    imp_df = pd.DataFrame({'特征': feature_cols, '重要性': model.feature_importances_})
    imp_df = imp_df.sort_values('重要性', ascending=True).tail(15)
    colors = ['#378ADD' if v > 0.05 else '#85B7EB' for v in imp_df['重要性']]
    ax.barh(imp_df['特征'], imp_df['重要性'], color=colors)
    ax.set_title(f'{name} 特征重要性', fontsize=13)
    ax.set_xlabel('重要性')
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(BASE_DIR, 'ml_feature_importance.png'), dpi=150, bbox_inches='tight')
print('Feature importance chart saved to ml_feature_importance.png')
# ========== 6. 模型对比 ==========
print("6. 模型综合效果对比")
print('\n【分类性能对比】')
class_perf = pd.DataFrame({
    '模型': model_eval_names,
    'Accuracy': [results[n]['accuracy'] for n in model_eval_names],
    'AUC': [results[n]['auc'] for n in model_eval_names],
})
print(class_perf.to_string(index=False))
print('\n【回测性能对比】')
print(summary_df.to_string(index=False))
print('\n【特征重要性排名 (Top 10)】')
for name in tree_models:
    if name in results and hasattr(results[name]['model'], 'feature_importances_'):
        fi = results[name]['model'].feature_importances_
        fi_dict = dict(zip(feature_cols, fi))
        fi_sorted = sorted(fi_dict.items(), key=lambda x: x[1], reverse=True)[:10]
        print(f'\n{name}:')
        for feat, val in fi_sorted:
            print(f'  {feat}: {val:.4f}')
best_sharpe_name = max(metrics.keys(), key=lambda k: metrics[k]['sharpe'])
best_auc_name = max(results.keys(), key=lambda k: results[k]['auc'])
print(f'\n- Best Sharpe: {best_sharpe_name} (Sharpe={metrics[best_sharpe_name]["sharpe"]:.2f})')
print(f'- Best AUC: {best_auc_name} (AUC={results[best_auc_name]["auc"]:.4f})')

[Section 2 executed - see HTML dashboard for full results]


## 3. 划分训练集/测试集 & 训练模型

In [ ]:
export_data = {
    'models': {},
    'metrics': {},
    'feature_importance': {},
    'quarterly_returns': {},
    'feature_cols': feature_cols,
}
for name, res in results.items():
    export_data['models'][name] = {
        'accuracy': res['accuracy'],
        'auc': res['auc'],
    }
for name, m in metrics.items():
    export_data['metrics'][name] = {
        'total_return': m['total_return'],
        'annual_return': m['annual_return'],
        'annual_vol': m['annual_vol'],
        'sharpe': m['sharpe'],
        'max_drawdown': m['max_drawdown'],
        'calmar': m['calmar'],
        'win_rate': m['win_rate'],
        'profit_loss_ratio': m['profit_loss_ratio'],
        'avg_return_per_quarter': m['avg_return_per_quarter'],
        'nav': list(m['nav']),
        'returns': list(m['returns']),
        'drawdown': list(m['drawdown']),
        'dates': [str(d) for d in m['dates']],
    }
for name in tree_models:
    if name in results and hasattr(results[name]['model'], 'feature_importances_'):
        fi = results[name]['model'].feature_importances_
        fi_dict = dict(zip(feature_cols, fi))
        fi_sorted = dict(sorted(fi_dict.items(), key=lambda x: x[1], reverse=True)[:15])
        export_data['feature_importance'][name] = fi_sorted
for name, qret in strategy_returns.items():
    export_data['quarterly_returns'][name] = {str(d): v for d, v in qret.items()}
def convert_native(obj):
    """Convert numpy types to Python native for JSON serialization"""
    if isinstance(obj, (np.integer,)):
        return int(obj)
    elif isinstance(obj, (np.floating,)):
        return float(obj)
    elif isinstance(obj, (np.ndarray,)):
        return [convert_native(x) for x in obj]
    elif isinstance(obj, dict):
        return {k: convert_native(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [convert_native(x) for x in obj]
    elif isinstance(obj, (np.bool_,)):
        return bool(obj)
    return obj
with open(os.path.join(BASE_DIR, 'ml_strategy_results.json'), 'w', encoding='utf-8') as f:
    json.dump(convert_native(export_data), f, ensure_ascii=False, indent=2)
print('\nResults saved to ml_strategy_results.json')
print('Done!')

[Section 3 executed - see HTML dashboard for full results]
